In [ ]:
import pandas as pd
import numpy as np
import datetime
import gc
import xgboost as xgb
import os
from warnings import filterwarnings
filterwarnings('ignore')

# --------------------------
# 全局配置：缓存+时间窗口+分片大小
# --------------------------
# 缓存路径（避免重复加载，提升效率）
CACHE_USER_ALL = "data/8/cache_user_all.pkl"  # 用户行为缓存
CACHE_ITEM_P = "data/8/cache_item_p.pkl"     # 商品子集P缓存
CACHE_TRAIN_SET = "data/8/cache_train_set.pkl"# 训练集缓存
CACHE_TEST_SET = "data/8/cache_test_set.pkl"  # 测试集缓存
USE_CACHE = True  # 缓存开关：True=复用，False=重新生成

# 时间窗口（匹配赛题11.18-12.18训练，12.19预测）
FEATURE_EXTRACTION_SLOT = 7  # 特征提取窗口：7天（捕捉近期偏好）
TRAIN_START_DATE = datetime.datetime(2014, 11, 18)  # 训练起始时间
LAST_BEHAVIOR_DATE = datetime.datetime(2014, 12, 18)# 最后行为日期（测试集基础日期）
PREDICT_DATE = datetime.datetime(2014, 12, 19)      # 预测日期

# 数据分片（处理千万级数据，避免内存溢出）
CHUNK_SIZE = 20_000_000     # 用户行为数据分片大小（2000万/片）
FEAT_CHUNK_SIZE = 5_000_000 # 特征计算分片大小（500万/片）

# 数据路径（赛题提供的原始数据）
USER_PART_A_PATH = "data/8/tianchi_fresh_comp_train_user_online_partA.txt"
USER_PART_B_PATH = "data/8/tianchi_fresh_comp_train_user_online_partB.txt"
ITEM_PATH = "data/8/tianchi_fresh_comp_train_item_online.txt"
OUTPUT_PATH = "data/8/tianchi_prediction_1219.txt"  # 输出路径

In [ ]:
def load_and_merge_data_with_cache():
    print("=== 开始加载原始数据（支持缓存复用）===")
    
    # 1. 加载商品子集P（缓存优化）
    if USE_CACHE and os.path.exists(CACHE_ITEM_P):
        print(f"加载商品缓存：{CACHE_ITEM_P}")
        item_p = pd.read_pickle(CACHE_ITEM_P)
    else:
        print("生成商品数据缓存...")
        item_cols = ['item_id', 'item_geohash', 'item_category']  # 赛题字段
        item_p = pd.read_csv(ITEM_PATH, sep='\t', names=item_cols)
        item_p = optimize_data_types(item_p)  # 优化数据类型（如uint16减少内存）
        item_p.to_pickle(CACHE_ITEM_P)
        print(f"商品缓存已保存：{CACHE_ITEM_P}")
    
    # 筛选有效商品ID（仅保留P中的商品）
    valid_item_ids = set(item_p['item_id'].astype(str))
    print(f"商品子集P规模：{item_p.shape}（{len(valid_item_ids)}个有效商品）")
    
    # 2. 加载用户行为数据（分Chunk+缓存）
    if USE_CACHE and os.path.exists(CACHE_USER_ALL):
        print(f"\n加载用户行为缓存：{CACHE_USER_ALL}")
        user_all = pd.read_pickle(CACHE_USER_ALL)
    else:
        print("生成用户行为数据缓存...")
        user_cols = ['user_id', 'item_id', 'behavior_type', 'user_geohash', 'item_category', 'time']
        user_chunks = []
        
        # 加载PartA（分Chunk避免内存溢出）
        for i, chunk in enumerate(pd.read_csv(USER_PART_A_PATH, sep='\t', names=user_cols, chunksize=CHUNK_SIZE)):
            chunk = _process_user_chunk(chunk, valid_item_ids)  # 筛选P中的商品行为
            if not chunk.empty:
                user_chunks.append(chunk)
            print(f"PartA加载：{i+1}块，累计{sum(len(c) for c in user_chunks):,}行")
        
        # 加载PartB（同上）
        for i, chunk in enumerate(pd.read_csv(USER_PART_B_PATH, sep='\t', names=user_cols, chunksize=CHUNK_SIZE)):
            chunk = _process_user_chunk(chunk, valid_item_ids)
            if not chunk.empty:
                user_chunks.append(chunk)
            print(f"PartB加载：{i+1}块，累计{sum(len(c) for c in user_chunks):,}行")
        
        # 合并并缓存
        user_all = pd.concat(user_chunks, ignore_index=True)
        user_all.to_pickle(CACHE_USER_ALL)
        print(f"用户缓存已保存：{CACHE_USER_ALL}")
    
    # 数据校验（确保行为类型完整：1=浏览，2=收藏，3=加购，4=购买）
    behavior_dist = user_all['behavior_type'].value_counts().sort_index()
    print(f"\n行为分布：{behavior_dist.to_dict()}（4=购买，占比{behavior_dist.get(4,0)/len(user_all):.2%}）")
    print(f"用户数据规模：{user_all.shape}，时间范围：{user_all['daystime'].min()}~{user_all['daystime'].max()}")
    return user_all, item_p

def _process_user_chunk(chunk, valid_item_ids):
    """处理单块用户行为：筛选P商品、解析时间、优化类型"""
    chunk['item_id'] = chunk['item_id'].astype(str)
    chunk = chunk[chunk['item_id'].isin(valid_item_ids)]  # 仅保留P中的商品
    if chunk.empty:
        return chunk
    # 解析时间（拆分日期和小时）
    chunk['daystime'] = pd.to_datetime(chunk['time'].str.split(' ').str[0])
    chunk['hours'] = chunk['time'].str.split(' ').str[1].astype(np.uint8)
    chunk = chunk.drop(columns=['time'])
    # 筛选时间范围（仅保留11.18后的数据）
    chunk = chunk[chunk['daystime'] >= TRAIN_START_DATE]
    # 优化数据类型（如user_id用uint32，减少内存）
    chunk = optimize_data_types(chunk)
    return chunk

def optimize_data_types(df):
    """优化数据类型，减少内存占用（关键：处理千万级数据必做）"""
    # 整数列压缩（如uint16适配0-65535，覆盖item_id、category等）
    int_cols = df.select_dtypes(include=['int64', 'int32']).columns
    for col in int_cols:
        col_max = df[col].max()
        if col_max <= 65535:
            df[col] = df[col].astype(np.uint16)
        else:
            df[col] = df[col].astype(np.uint32)
    # 字符串列转为category（如user_geohash，减少重复存储）
    str_cols = df.select_dtypes(include=['object']).columns
    for col in str_cols:
        if df[col].nunique() < 100000:
            df[col] = df[col].astype('category')
    # 删除无用列（如user_geohash为空率高，不影响购买预测）
    df = df.drop(columns=['user_geohash', 'item_geohash'], errors='ignore')
    return df

# 执行数据加载
user_all, item_p = load_and_merge_data_with_cache()

In [ ]:
def _merge_features(base_df, feat_data, prev_day):
    """合并4类核心特征：用户+商品+品类+交叉"""
    # 1. 用户特征：反映用户购买能力与活跃度
    # user_act_1-4：用户4类行为的总次数（1=浏览，2=收藏，3=加购，4=购买）
    feat_user = feat_data.groupby('user_id')['behavior_type'].value_counts().unstack(fill_value=0).reset_index()
    for btn in [1,2,3,4]:
        if btn not in feat_user.columns:
            feat_user[btn] = 0
    feat_user.columns = ['user_id'] + [f'user_act_{btn}' for btn in [1,2,3,4]]
    # user_active_days：用户在特征窗口内的活跃天数（活跃天数越多，购买概率越高）
    feat_user_liveday = feat_data.groupby('user_id')['daystime'].nunique().reset_index(name='user_active_days').astype({'user_active_days': np.uint8})
    # 合并用户特征
    base_df = base_df.merge(feat_user, on='user_id', how='left')
    base_df = base_df.merge(feat_user_liveday, on='user_id', how='left')
    
    # 2. 商品特征：反映商品受欢迎程度
    # item_act_1-4：商品被所有用户的4类行为次数（加购/收藏多的商品更易被购买）
    feat_item = feat_data.groupby('item_id')['behavior_type'].value_counts().unstack(fill_value=0).reset_index()
    for btn in [1,2,3,4]:
        if btn not in feat_item.columns:
            feat_item[btn] = 0
    feat_item.columns = ['item_id'] + [f'item_act_{btn}' for btn in [1,2,3,4]]
    base_df = base_df.merge(feat_item, on='item_id', how='left')
    
    # 3. 品类特征：反映品类热度
    # cate_act_1-4：品类下所有商品的4类行为次数（热门品类更易转化）
    feat_cate = feat_data.groupby('item_category')['behavior_type'].value_counts().unstack(fill_value=0).reset_index()
    for btn in [1,2,3,4]:
        if btn not in feat_cate.columns:
            feat_cate[btn] = 0
    feat_cate.columns = ['item_category'] + [f'cate_act_{btn}' for btn in [1,2,3,4]]
    base_df = base_df.merge(feat_cate, on='item_category', how='left')
    
    # 4. 交叉特征：反映用户对特定商品/品类的偏好
    # ui_act_1-4：用户对该商品的4类行为次数（用户重复交互的商品更易购买）
    feat_ui = chunked_groupby_feat(feat_data, ['user_id', 'item_id'], 'behavior_type', 'ui')
    # uc_act_1-4：用户对该品类的4类行为次数（用户偏好的品类更易购买）
    feat_uc = chunked_groupby_feat(feat_data, ['user_id', 'item_category'], 'behavior_type', 'uc')
    base_df = base_df.merge(feat_ui, on=['user_id', 'item_id'], how='left')
    base_df = base_df.merge(feat_uc, on=['user_id', 'item_category'], how='left')
    
    # 填充缺失值（无行为则为0）+ 优化类型
    base_df = base_df.fillna(0)
    return optimize_data_types(base_df)

def chunked_groupby_feat(data, group_keys, target_col, feat_prefix):
    """分Chunk计算交叉特征（避免大表groupby内存溢出）"""
    chunks = []
    for i in range(0, len(data), FEAT_CHUNK_SIZE):
        chunk = data.iloc[i:i+FEAT_CHUNK_SIZE]
        # 计算单Chunk的groupby特征
        chunk_feat = chunk.groupby(group_keys)[target_col].value_counts().unstack(fill_value=0).reset_index()
        for btn in [1,2,3,4]:
            if btn not in chunk_feat.columns:
                chunk_feat[btn] = 0
        # 重命名列（如ui_act_1）
        chunk_feat.columns = list(group_keys) + [f'{feat_prefix}_act_{btn}' for btn in [1,2,3,4]]
        chunks.append(chunk_feat)
        del chunk
        gc.collect()
    
    # 合并多Chunk结果
    if len(chunks) == 1:
        return chunks[0]
    merged = chunks[0]
    for chunk in chunks[1:]:
        merged = pd.merge(merged, chunk, on=group_keys, how='outer', suffixes=('', '_dup'))
        # 合并重复列（如ui_act_1和ui_act_1_dup求和）
        for btn in [1,2,3,4]:
            col = f'{feat_prefix}_act_{btn}'
            dup_col = f'{feat_prefix}_act_{btn}_dup'
            if dup_col in merged.columns:
                merged[col] = merged[col].fillna(0) + merged[dup_col].fillna(0)
                merged = merged.drop(columns=dup_col)
        merged = merged.fillna(0)
    
    # 优化数据类型
    for btn in [1,2,3,4]:
        col = f'{feat_prefix}_act_{btn}'
        merged[col] = merged[col].astype(np.uint16)
    return merged

def build_train_set_with_cache(user_all):
    print("\n=== 开始处理训练集（支持缓存复用）===")
    # 复用缓存
    if USE_CACHE and os.path.exists(CACHE_TRAIN_SET):
        print(f"加载训练集缓存：{CACHE_TRAIN_SET}")
        train_set = pd.read_pickle(CACHE_TRAIN_SET)
        # 校验缓存有效性（确保有label列和正样本）
        if 'label' not in train_set.columns or train_set['label'].sum() == 0:
            raise ValueError("训练集缓存损坏，删除后重新运行！")
        return train_set
    
    # 无缓存时，生成训练集（滑动窗口策略）
    print("生成训练集（无缓存，首次运行较慢）...")
    # 标签窗口：取最后5个有效日期（12.14-12.18），避免时间过早的样本
    label_days = [LAST_BEHAVIOR_DATE - datetime.timedelta(days=i) for i in range(5)]
    label_days = [d for d in label_days if d >= TRAIN_START_DATE + datetime.timedelta(days=FEATURE_EXTRACTION_SLOT)]
    train_set_list = []
    
    for idx, end_time in enumerate(label_days):
        # 特征窗口：end_time前7天（FEATURE_EXTRACTION_SLOT=7）
        feat_start = end_time - datetime.timedelta(days=FEATURE_EXTRACTION_SLOT)
        train_window = user_all[(user_all['daystime'] >= feat_start) & (user_all['daystime'] <= end_time)]
        # 基础样本：end_time前一天的交互（用户对商品的近期行为）
        prev_day = end_time - datetime.timedelta(days=1)
        base_train = train_window[train_window['daystime'] == prev_day].drop_duplicates(['user_id', 'item_id'])
        if base_train.empty:
            print(f"跳过窗口 {end_time.strftime('%Y-%m-%d')}：无交互样本")
            continue
        
        # 正样本：end_time当天的购买行为（需去重，一个用户-商品对仅算一次购买）
        buy_events = train_window[(train_window['daystime'] == end_time) & (train_window['behavior_type'] == 4)].drop_duplicates(['user_id', 'item_id'])
        print(f"窗口 {end_time.strftime('%Y-%m-%d')}：交互{len(base_train):,}行，购买{len(buy_events):,}行")
        
        # 打标：构建用户-商品唯一键（ui_key），匹配购买行为
        base_train['ui_key'] = base_train['user_id'].astype(str) + '_' + base_train['item_id'].astype(str)
        buy_keys = set()
        if not buy_events.empty:
            buy_events['ui_key'] = buy_events['user_id'].astype(str) + '_' + buy_events['item_id'].astype(str)
            buy_keys = set(buy_events['ui_key'])
        base_train['label'] = base_train['ui_key'].isin(buy_keys).astype(np.uint8)  # 1=购买，0=不购买
        
        # 合并特征（用户/商品/品类/交叉特征）
        base_train = _merge_features(base_train, train_window, prev_day)
        base_train = base_train.drop(columns=['ui_key'], errors='ignore')
        train_set_list.append(base_train)
        
        # 释放内存
        del train_window, base_train, buy_events
        gc.collect()
    
    # 合并训练集并缓存
    train_set = pd.concat(train_set_list, axis=0, ignore_index=True)
    train_set.to_pickle(CACHE_TRAIN_SET)
    print(f"\n训练集缓存已保存：{CACHE_TRAIN_SET}")
    # 统计信息（正样本占比低，后续需平衡）
    pos_count = train_set['label'].sum()
    neg_count = len(train_set) - pos_count
    print(f"训练集规模：{train_set.shape}，正样本{pos_count:,}（{pos_count/len(train_set):.2%}），负样本{neg_count:,}")
    return train_set

In [ ]:
def balance_train_set(train_set):
    """样本平衡：负样本抽样（正负比1:30），避免模型偏向负样本"""
    print("\n=== 开始样本平衡 ===\n")
    train_pos = train_set[train_set['label'] == 1].copy()  # 正样本
    train_neg = train_set[train_set['label'] == 0].copy()  # 负样本
    pos_count = len(train_pos)
    neg_count = len(train_neg)
    
    if pos_count == 0:
        raise ValueError("无正样本，无法训练！")
    
    # 负样本抽样：最多取正样本的30倍（过多负样本会稀释正样本信号）
    sample_neg_size = min(neg_count, pos_count * 30)
    train_neg_sampled = train_neg.sample(n=sample_neg_size, random_state=10, replace=False)
    
    # 合并平衡后的样本
    balanced_train = pd.concat([train_pos, train_neg_sampled], axis=0, ignore_index=True)
    balanced_pos = balanced_train['label'].sum()
    balanced_neg = len(balanced_train) - balanced_pos
    print(f"平衡前：正样本{pos_count:,}，负样本{neg_count:,}，正负比1:{neg_count/pos_count:.0f}")
    print(f"平衡后：正样本{balanced_pos:,}，负样本{balanced_neg:,}，正负比1:{balanced_neg/balanced_pos:.0f}")
    return balanced_train

def train_and_predict(balanced_train, base_test):
    """XGBoost训练与预测：输出购买概率TopN的用户-商品对"""
    print("\n=== 开始模型训练与预测 ===\n")
    # 筛选特征列（排除ID、时间、标签等非特征列）
    drop_cols = ['user_id', 'item_id', 'item_category', 'label', 'hours', 'daystime']
    train_features = balanced_train.drop(columns=[c for c in drop_cols if c in balanced_train.columns])
    train_labels = balanced_train['label'].values
    test_features = base_test.drop(columns=[c for c in drop_cols if c in base_test.columns and c != 'label'])
    
    # 确保训练/测试特征一致
    common_cols = list(set(train_features.columns) & set(test_features.columns))
    if not common_cols:
        raise ValueError("无共同特征！")
    train_features = train_features[common_cols]
    test_features = test_features[common_cols]
    print(f"最终使用特征数：{len(common_cols)}（如user_act_4、ui_act_3等）")
    
    # XGBoost参数（适配二分类推荐任务）
    params = {
        'max_depth': 5,          # 树深：避免过拟合（移动推荐特征维度不高）
        'colsample_bytree': 0.7, # 特征采样：减少冗余特征影响
        'subsample': 0.7,        # 样本采样：提升泛化能力
        'eta': 0.04,            # 学习率：小步迭代，避免震荡
        'objective': 'binary:logistic', # 二分类（预测购买概率）
        'eval_metric': 'auc',    # 评估指标：AUC（适合不平衡数据）
        'min_child_weight': 4,   # 最小子节点权重：避免过拟合
        'seed': 10,             # 随机种子：结果可复现
        'nthread': -1           # 多线程：利用所有CPU核心
    }
    num_round = 800  # 迭代轮次：通过验证集调优确定
    
    # 训练模型
    # dtrain = xgb.DMatrix(train_features, label=train_labels)
    # dtest = xgb.DMatrix(test_features)
    # print(f"训练XGBoost模型（{num_round}轮迭代）...")
    # bst = xgb.train(list(params.items()), dtrain, num_round)
    from pytabkit import LGBM_TD_Classifier
    model = LGBM_TD_Classifier(
        verbosity=1,
        random_state=10,
        val_metric_name='1-auc_ovr',
    )
    model.fit(train_features, train_labels)
    test_proba = model.predict(test_features)
    
    # 预测购买概率并排序
    # test_proba = bst.predict(dtest)  # 输出0-1的购买概率
    # 生成结果：按概率降序，取前10000条（避免输出过多，符合推荐TopN逻辑）
    result = pd.DataFrame({
        'user_id': base_test['user_id'].astype(str),
        'item_id': base_test['item_id'].astype(str),
        'buy_prob': test_proba
    }).sort_values('buy_prob', ascending=False).reset_index(drop=True)
    
    # 输出赛题要求格式（Tab分隔，无表头）
    result[['user_id', 'item_id']].to_csv(OUTPUT_PATH, sep='\t', index=False, header=False)
    print(f"\n预测结果已保存：{OUTPUT_PATH}")
    print(f"输出购买对数量：{len(result)}（按购买概率降序）")
    return result

def build_test_set_with_cache(user_all, item_p):
    print("\n=== 开始处理测试集（支持缓存复用）===")
    if USE_CACHE and os.path.exists(CACHE_TEST_SET):
        print(f"加载测试集缓存：{CACHE_TEST_SET}")
        base_test = pd.read_pickle(CACHE_TEST_SET)
        return base_test
    
    # 生成测试集：12.18的交互样本（预测12.19的购买）
    print("生成测试集...")
    # 特征窗口：12.11-12.18（7天）
    test_feat_window = user_all[
        (user_all['daystime'] >= LAST_BEHAVIOR_DATE - datetime.timedelta(days=FEATURE_EXTRACTION_SLOT)) &
        (user_all['daystime'] <= LAST_BEHAVIOR_DATE)
    ]
    # 基础样本：12.18的交互（仅保留P中的商品）
    base_test = user_all[user_all['daystime'] == LAST_BEHAVIOR_DATE].drop_duplicates(['user_id', 'item_id'])
    base_test = base_test[base_test['item_id'].isin(item_p['item_id'].astype(str))]
    if base_test.empty:
        raise ValueError("测试集无样本！")
    
    # 合并特征（同训练集特征逻辑，确保特征一致性）
    base_test = _merge_features(base_test, test_feat_window, LAST_BEHAVIOR_DATE)
    # 缓存测试集
    base_test.to_pickle(CACHE_TEST_SET)
    print(f"测试集缓存已保存：{CACHE_TEST_SET}")
    # print(f"测试集规模：{base_test.shape}（{len(base_test['user_id'].nunique())}个用户，{len(base_test['item_id'].nunique())}个商品）")
    return base_test

# 构建训练集
train_set = build_train_set_with_cache(user_all)
# 执行样本平衡与训练
balanced_train = balance_train_set(train_set)
# 构建测试集（逻辑类似训练集，用12.18的交互作为基础样本）
base_test = build_test_set_with_cache(user_all, item_p)
# 模型训练与预测
final_result = train_and_predict(balanced_train, base_test)


=== 开始处理训练集（支持缓存复用）===
加载训练集缓存：data/8/cache_train_set.pkl

=== 开始样本平衡 ===

平衡前：正样本28,951，负样本7,599,715，正负比1:263
平衡后：正样本28,951，负样本868,530，正负比1:30

=== 开始处理测试集（支持缓存复用）===
加载测试集缓存：data/8/cache_test_set.pkl

=== 开始模型训练与预测 ===

最终使用特征数：22（如user_act_4、ui_act_3等）
Columns classified as continuous: ['user_act_4', 'uc_act_3', 'user_act_2', 'behavior_type', 'ui_act_1', 'ui_act_4', 'cate_act_3', 'user_act_1', 'ui_act_2', 'cate_act_2', 'item_act_1', 'uc_act_4', 'item_act_2', 'item_act_3', 'user_active_days', 'ui_act_3', 'user_act_3', 'uc_act_1', 'cate_act_1', 'cate_act_4', 'uc_act_2', 'item_act_4']
Columns classified as categorical: []
[LightGBM] [Info] Number of positive: 26056, number of negative: 781677
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055145 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3050
[LightGBM] [Info] Number of data poin

围绕“用户—商户”的转化预测，整体采用“时序滑窗构造样本 + 多粒度统计/比例 + OOF 目标编码 + GBDT 系列模型融合”的路径。在实现上严格按时间切分，避免信息泄露。
数据切分与样本构造
时间切分：按天/小时做前闭后开的滑窗，训练用历史窗口，验证/线上用未来窗口，确保无“时间穿越”。
负采样与去偏：对未转化对进行下采样，按商户/用户活跃度、行为类型分层采样，缓解类别极不平衡。
候选召回：基于用户近期交互的商户集合，结合同类目/同品牌的近邻扩展，兼顾召回率与计算开销。
特征工程
基础特征：
用户：全局活跃度、最近活跃天数、行为计数（点击/收藏/加购/购买）、去重交互实体数（商户/品牌/类目/商品），多尺度窗口（1/3/7/14/30天）。
商家：行为计数分布，分时段热度曲线。
用户与商户：历史交互次数、最近一次/最近k次交互间隔、最近交互行为类型、历史转化标记，结合时间与行为类型特征。
比例特征：
用户与商家分行为占比：点击，收藏，加购，购买各个行为在各个窗口的比例。
时间分布占比：不同时段的行为占比，同一窗口内的不同行为占比。
不同窗口对比：不同时间窗口的行为占比变化率。
防泄漏策略
样本切分（训练集，验证集。编码集），目标比例用编码段估计，仅向未来传播，杜绝泄露与拟合。，杜绝泄露与过拟合。
模型训练
模型：Lightgbm与Xgboost为主，根据结果手动小范围调参（树深，叶子数，学习率，L1,L2正则等）。未做复杂CV。

tabm-mini: [ 0  0  5  0  0  2  4  4 21  0]

21 -> self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00019936661888626597), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(864), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(27)}

tabm [ 2  0  0  0  0  9  3  0 19  0]

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005058942529815513), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(480), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(96)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003370062728089647), 'weight_decay': np.float64(0.00025518969970339756), 'n_blocks': np.int64(5), 'd_block': np.int64(976), 'dropout': np.float64(0.1255057131396986), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(23)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0021399752156178284), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(688), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(84)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0011136343791654155), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(160), 'dropout': np.float64(0.3109730730172545), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(96)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003706377021434626), 'weight_decay': np.float64(0.0674066641717872), 'n_blocks': np.int64(5), 'd_block': np.int64(384), 'dropout': np.float64(0.37362818327260205), 'num_emb_type': 'pwl', 'd_embedding': np.int64(24), 'num_emb_n_bins': np.int64(79)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00013313089945964177), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(576), 'dropout': np.float64(0.4127963921140429), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(83)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.000728482449630055), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(800), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(55)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005420969545908613), 'weight_decay': np.float64(0.028783143186014805), 'n_blocks': np.int64(3), 'd_block': np.int64(896), 'dropout': np.float64(0.14411909219317454), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(42)}

19 -> self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00019936661888626597), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(864), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(27)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0007561404492770056), 'weight_decay': np.float64(0.022028699054526424), 'n_blocks': np.int64(2), 'd_block': np.int64(384), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(112)}